# MongoDB Vector Search

Un espacio para **jugar** con todo lo del proyecto: la conexión, los embeddings, `$vectorSearch`, `$search`,
`$rankFusion`, tus propias colecciones y el RAG. Las lecciones (`lessons/`) muestran cada tema en orden; aquí puedes
cambiar cualquier cosa y volver a ejecutar.

**Cómo usarlo**

1. Ejecuta las celdas de **Configuración** (secciones 0 y 1) una vez.
2. Después salta a la sección que quieras. Las celdas marcadas con **Juega aquí** están hechas para que las
   cambies: consultas, filtros, frases, pipelines.
3. Nada de lo que hagas aquí rompe el proyecto: la sección 8 usa su propia colección (`sandbox`), y si alteras
   `chunks`, `python lessons/00_seed.py` la deja como al principio en segundos.

> Requisito: haber ejecutado `python lessons/00_seed.py` al menos una vez.

| # | Sección | Qué puedes probar |
| --- | --- | --- |
| 0 | Configuración | Conexión, estado de la base y de los índices |
| 1 | Herramientas | Funciones `buscar`, `leer`, `similitud`, `comparar`, `tabla` |
| 2 | Explorar la colección | `find`, `aggregate`, proyecciones |
| 3 | Embeddings | Similitud entre tus frases, prefijos, mapa de calor |
| 4 | Búsqueda semántica | `$vectorSearch`, filtros, `numCandidates`, exacta vs aproximada |
| 5 | Texto completo e híbrida | `$search`, operadores Lucene, `$rankFusion` con pesos |
| 6 | Pipeline libre | Cualquier agregación que empiece con una búsqueda |
| 7 | Mapa de embeddings | Los vectores proyectados en 2D, con tu pregunta encima |
| 8 | Tu propia colección | Tus textos, tu índice vectorial, tus búsquedas |
| 9 | RAG | Preguntas al libro, con o sin Claude |

## 0. Configuración

La conexión sale de `MONGO_URI` (`.envs/mongo.env`). Dentro del contenedor es
`mongodb://mongodb:27017/?directConnection=true`; el notebook usa las mismas funciones que las lecciones
(`src/lab/`).

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pymongo.operations import SearchIndexModel

from lab import book, rag
from lab.client import get_client
from lab.config import EMBEDDING_MODEL, MONGO_DB, MONGO_URI, TEXT_INDEX, VECTOR_INDEX
from lab.embeddings import dimensions, embed_passages, embed_query, get_model
from lab.search import (FIELDS, SEARCHES, full_text, hybrid, reference, search_indexes, semantic, text_stage,
                        vector_index, vector_stage, wait_until_queryable)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

client = get_client()
db = client[MONGO_DB]
chunks = db.chunks

print("URI:        ", MONGO_URI)
print("MongoDB:    ", client.server_info()["version"])
print("Base:       ", MONGO_DB, "| colecciones:", db.list_collection_names())
print("Fragmentos: ", chunks.estimated_document_count())
print("Modelo:     ", EMBEDDING_MODEL, f"({dimensions()} dimensiones)")
pd.DataFrame([{"índice": i["name"], "status": i["status"], "queryable": i["queryable"], "docs": i.get("numDocs")}
              for i in search_indexes(chunks).values()])

## 1. Herramientas

Funciones cortas para no repetir código. Están aquí, a la vista, para que las leas y las cambies.

| Función | Qué hace |
| --- | --- |
| `tabla(cursor_o_lista)` | Convierte resultados de MongoDB en un DataFrame (sin el vector) |
| `buscar(consulta, modo, k, filtro)` | Búsqueda `semantic`, `text` o `hybrid`; la consulta puede ser texto o un vector |
| `leer(_id)` | Imprime el texto completo de un fragmento |
| `similitud(textos, pregunta)` | Similitud coseno entre tus frases (y contra una pregunta) |
| `comparar(consulta)` | Top k de las tres búsquedas lado a lado |

In [ ]:
def tabla(rows, sin=("embedding",)):
    """Resultados de MongoDB -> DataFrame, sin las columnas pesadas."""
    df = pd.DataFrame(list(rows))
    return df.drop(columns=[c for c in sin if c in df.columns])


def buscar(consulta, modo="semantic", k=5, filtro=None):
    """Busca en chunks. modo: semantic | text | hybrid. filtro: p. ej. {"volume": 2}."""
    resultados = SEARCHES[modo](chunks, consulta, k=k, filter=filtro)
    return pd.DataFrame([{"_id": d["_id"], "ref": reference(d), "score": round(d["score"], 4),
                          "capítulo": d["chapter_title"][:60], "texto": d["text"][:160].replace("\n", " ") + "…"}
                         for d in resultados])


def leer(_id):
    d = chunks.find_one({"_id": _id})
    print(f"[{reference(d)}] {d['chapter_title']}\n")
    print(d["text"])


def similitud(textos, pregunta=None):
    """Matriz de similitud coseno entre textos (vectores normalizados: coseno = producto punto)."""
    vectores = np.array(embed_passages(textos))
    df = pd.DataFrame(vectores @ vectores.T, index=textos, columns=[t[:25] for t in textos]).round(3)
    if pregunta:
        df.insert(0, f"vs '{pregunta[:25]}'", (vectores @ np.array(embed_query(pregunta))).round(3))
    return df


def comparar(consulta, k=5, filtro=None):
    columnas = {modo: [reference(d) for d in fn(chunks, consulta, k=k, filter=filtro)] for modo, fn in SEARCHES.items()}
    return pd.DataFrame({m: v + [""] * (k - len(v)) for m, v in columnas.items()}, index=range(1, k + 1))


buscar("don Quijote confunde unos molinos con gigantes", k=3)

## 2. Explorar la colección

Consultas normales de MongoDB (MQL): las resuelve `mongod` con sus índices B-tree, sin pasar por `mongot`.

In [ ]:
# Un documento completo (el vector recortado para que se pueda leer)
doc = chunks.find_one({"_id": "v1-c008-000"})
{**doc, "embedding": doc["embedding"][:5] + ["…"], "text": doc["text"][:200] + "…"}

In [ ]:
# Fragmentos y largo promedio por volumen
tabla(chunks.aggregate([
    {"$group": {"_id": "$volume", "fragmentos": {"$sum": 1}, "capítulos": {"$addToSet": "$chapter"},
                "largo_promedio": {"$avg": {"$strLenCP": "$text"}}}},
    {"$project": {"volumen": "$_id", "_id": 0, "fragmentos": 1, "capítulos": {"$size": "$capítulos"},
                  "largo_promedio": {"$round": ["$largo_promedio", 0]}}},
    {"$sort": {"volumen": 1}},
]))

### Juega aquí: cualquier `find` o `aggregate`

In [ ]:
filtro = {"volume": 1, "chapter": 8}                     # cambia el filtro
proyeccion = {"_id": 1, "position": 1, "text": 1}
tabla(chunks.find(filtro, proyeccion).sort("position", 1).limit(5))

# Ideas:
#   {"text": {"$regex": "Rocinante"}}                    búsqueda por patrón (COLLSCAN, lenta y literal)
#   {"chapter_title": {"$regex": "molinos", "$options": "i"}}
#   chunks.count_documents({"volume": 2, "chapter": {"$gte": 70}})

## 3. Embeddings

Un embedding es un vector que representa el significado de un texto. Aquí puedes ver qué tan cerca quedan tus
frases entre sí y de una pregunta.

In [ ]:
frases = [
    "don Quijote arremete contra los molinos de viento",
    "el caballero ataca a unos gigantes imaginarios",
    "Sancho come pan y queso a la sombra de un árbol",
    "el escudero tiene hambre y busca algo de comer",
    "la factura de energía llegó más alta este mes",
]
similitud(frases, pregunta="una batalla contra enemigos que no existen")

In [ ]:
# La misma matriz como mapa de calor
m = similitud(frases)
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(m.values, cmap="viridis")
ax.set_xticks(range(len(frases)), [f[:20] for f in frases], rotation=45, ha="right")
ax.set_yticks(range(len(frases)), [f[:30] for f in frases])
for i in range(len(frases)):
    for j in range(len(frases)):
        ax.text(j, i, f"{m.values[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
fig.colorbar(im)
plt.tight_layout()

### Juega aquí: tus frases, en cualquier idioma

In [ ]:
mis_frases = [
    "el sol sale por el este",
    "the sun rises in the east",     # el modelo es multilingüe
    "amanece por el oriente",
    "el sol se pone por el oeste",
]
similitud(mis_frases, pregunta="¿por dónde sale el sol?")

# Ideas: antónimos (¿quedan cerca o lejos?), negaciones, nombres propios, frases muy largas vs muy cortas.

In [ ]:
# Un vector por dentro: largo (norma), primeros valores y prefijos E5
v = np.array(embed_query("don Quijote"))
print("dimensiones:", v.shape[0], "| norma:", round(float(np.linalg.norm(v)), 4))
print("primeros 10:", v[:10].round(4))

modelo = get_model()
con, sin = modelo.encode(["query: don Quijote", "don Quijote"], normalize_embeddings=True)
print("similitud entre 'query: don Quijote' y 'don Quijote':", round(float(con @ sin), 4))

## 4. Búsqueda semántica (`$vectorSearch`)

`buscar(...)` arma la etapa con `vector_stage` (en `src/lab/search.py`). Aquí la tienes a mano para cambiar cada
parámetro.

In [ ]:
buscar("Sancho se convierte en gobernador de una isla")

In [ ]:
# Pre-filtros: solo campos declarados como "filter" en el índice (volume, chapter)
buscar("una batalla", filtro={"$and": [{"volume": 1}, {"chapter": {"$gte": 15, "$lte": 25}}]})

In [ ]:
# Ver el texto completo de un resultado
leer("v1-c008-001")

### Juega aquí: la etapa `$vectorSearch` a mano

In [ ]:
consulta = "una dama encantada que en realidad es una labradora"
etapa = {"$vectorSearch": {
    "index": VECTOR_INDEX,
    "path": "embedding",
    "queryVector": embed_query(consulta),
    "numCandidates": 100,          # prueba 5, 10, 50, 500
    "limit": 5,                    # cuántos devuelve
    # "exact": True,               # búsqueda exacta: quita numCandidates si la activas
    # "filter": {"volume": 2},
}}
tabla(chunks.aggregate([etapa, {"$project": {**FIELDS, "score": {"$meta": "vectorSearchScore"}}}]))

In [ ]:
# ¿Cuánto pierde la búsqueda aproximada? recall@k contra la exacta, para varios numCandidates
def recall(consulta, k=10, candidatos=(10, 20, 50, 100, 400)):
    vector = embed_query(consulta)
    ids = lambda etapa: {d["_id"] for d in chunks.aggregate([etapa, {"$project": {"_id": 1}}])}
    exacta = ids(vector_stage(vector, k=k, exact=True))
    return pd.DataFrame([{"numCandidates": c, f"recall@{k}": len(ids(vector_stage(vector, k=k, num_candidates=c)) & exacta) / k}
                         for c in candidatos])


recall("Sancho pierde a su asno")

In [ ]:
# "Más como éste": usar el vector de un fragmento guardado como consulta (sin el modelo)
referencia = chunks.find_one({"_id": "v2-c045-001"})
print(referencia["text"][:200], "…")
buscar(referencia["embedding"], filtro={"chapter": {"$ne": 45}})

## 5. Texto completo (`$search`) e híbrida (`$rankFusion`)

In [ ]:
comparar("¿Quién es Clavileño?")

### Juega aquí: operadores de `$search` (Lucene)

In [ ]:
operador = {"phrase": {"query": "caballero de la triste figura", "path": "text"}}
# Otros para probar:
#   {"text": {"query": "Dulsinea", "path": "text", "fuzzy": {"maxEdits": 1}}}        tolera errores de tipeo
#   {"wildcard": {"query": "encant*", "path": "text", "allowAnalyzedField": True}}
#   {"compound": {"must": [{"text": {"query": "Sancho", "path": "text"}}],
#                 "mustNot": [{"text": {"query": "ínsula", "path": "text"}}]}}
filas = []
for d in chunks.aggregate([
    {"$search": {"index": TEXT_INDEX, **operador, "highlight": {"path": "text"}}},
    {"$limit": 5},
    {"$project": {"_id": 1, "score": {"$meta": "searchScore"}, "resaltado": {"$meta": "searchHighlights"}}},
]):
    # searchHighlights: trozos del texto; los de type "hit" son las palabras encontradas (aquí entre [ ])
    trozo = d["resaltado"][0]["texts"] if d["resaltado"] else []
    filas.append({"_id": d["_id"], "score": round(d["score"], 3),
                  "resaltado": "".join(f"[{t['value']}]" if t["type"] == "hit" else t["value"] for t in trozo)[:150]})
pd.DataFrame(filas)

### Juega aquí: `$rankFusion` con pesos

In [ ]:
def hibrida(consulta, pesos=None, k=5, candidatos=20):
    etapa = {"$rankFusion": {"input": {"pipelines": {
        "semantic": [vector_stage(consulta, k=candidatos)],
        "text": [text_stage(consulta), {"$limit": candidatos}],
    }}, "scoreDetails": True}}
    if pesos:
        etapa["$rankFusion"]["combination"] = {"weights": pesos}
    filas = []
    for d in chunks.aggregate([etapa, {"$limit": k}, {"$project": {**FIELDS, "detalle": {"$meta": "scoreDetails"}}}]):
        ranks = {x["inputPipelineName"]: x.get("rank") for x in d["detalle"]["details"]}
        filas.append({"ref": reference(d), "score": round(d["detalle"]["value"], 5),
                      "rank_semántica": ranks.get("semantic"), "rank_texto": ranks.get("text"),
                      "capítulo": d["chapter_title"][:50]})
    return pd.DataFrame(filas)


consulta = "¿Qué le pasó a Sancho en la ínsula Barataria?"
display(hibrida(consulta))                                  # pesos iguales
display(hibrida(consulta, pesos={"semantic": 3, "text": 1}))  # cambia los pesos y compara

## 6. Pipeline libre

Después de `$vectorSearch` o `$search` el pipeline es una agregación normal: `$group`, `$lookup`, `$sort`,
`$facet`…

### Juega aquí: arma tu pipeline

In [ ]:
pipeline = [
    vector_stage("encantadores y hechiceros", k=50, num_candidates=500),
    {"$group": {"_id": {"volumen": "$volume", "capítulo": "$chapter"},
                "fragmentos": {"$sum": 1},
                "mejor_score": {"$max": {"$meta": "vectorSearchScore"}}}},
    {"$sort": {"fragmentos": -1, "mejor_score": -1}},
    {"$limit": 10},
]
tabla(chunks.aggregate(pipeline))

# Ideas:
#   - cambia vector_stage(...) por text_stage("Dulcinea") y agrega {"$limit": 50} después
#   - agrega {"$match": {"_id.volumen": 2}} después del $group
#   - $lookup a chunks para traer el fragmento siguiente (position + 1) de cada resultado

## 7. Mapa de embeddings

Los vectores tienen 384 dimensiones; con **PCA** (aquí con `numpy`, vía SVD) se proyectan en 2 para verlos. Cada
punto es un fragmento y el color es su capítulo. La ✕ roja es tu pregunta.

### Juega aquí: elige capítulos y una pregunta

In [ ]:
capitulos = [(1, 8), (1, 17), (2, 23), (2, 45), (2, 74)]   # (volumen, capítulo)
pregunta = "Sancho gobierna una isla"

docs = list(chunks.find({"$or": [{"volume": v, "chapter": c} for v, c in capitulos]},
                        {"volume": 1, "chapter": 1, "embedding": 1}))
X = np.array([d["embedding"] for d in docs])
centro = X.mean(axis=0)
_, _, Vt = np.linalg.svd(X - centro, full_matrices=False)
proyectar = lambda M: (np.atleast_2d(M) - centro) @ Vt[:2].T

puntos = proyectar(X)
fig, ax = plt.subplots(figsize=(8, 6))
for v, c in capitulos:
    sel = [i for i, d in enumerate(docs) if (d["volume"], d["chapter"]) == (v, c)]
    ax.scatter(puntos[sel, 0], puntos[sel, 1], label=f"vol. {v}, cap. {c}", alpha=0.7)
q = proyectar(np.array(embed_query(pregunta)))
ax.scatter(q[:, 0], q[:, 1], marker="x", s=200, c="red", label=f"'{pregunta}'")
ax.legend()
ax.set_title("Fragmentos proyectados en 2D (PCA)")
plt.tight_layout()

## 8. Tu propia colección

Una colección `sandbox` con **tus** textos: los conviertes en vectores, creas un índice vectorial y buscas. Así
se aplica todo el proyecto a cualquier otro tema (preguntas frecuentes, productos, notas, documentos…).

### Juega aquí: tus textos

In [ ]:
mis_textos = [
    {"categoria": "energía", "texto": "La energía solar convierte la luz del sol en electricidad con paneles fotovoltaicos."},
    {"categoria": "energía", "texto": "Un transformador cambia el nivel de voltaje de la corriente alterna."},
    {"categoria": "energía", "texto": "La demanda de potencia máxima define el cargo por capacidad en la factura."},
    {"categoria": "cocina", "texto": "Para hacer arepas se mezcla harina de maíz precocida con agua y sal."},
    {"categoria": "cocina", "texto": "El café se prepara mejor con agua a unos 92 grados."},
    {"categoria": "deporte", "texto": "En el fútbol cada equipo tiene once jugadores en la cancha."},
    {"categoria": "deporte", "texto": "El ciclismo de ruta en Colombia tiene etapas de alta montaña."},
]

sandbox = db.sandbox
sandbox.delete_many({})
vectores = embed_passages([t["texto"] for t in mis_textos])
sandbox.insert_many([{**t, "embedding": v} for t, v in zip(mis_textos, vectores)])
print("documentos:", sandbox.count_documents({}))

In [ ]:
# Índice vectorial con "categoria" como filtro (se crea una vez; mongot lo construye en segundos)
INDICE_SANDBOX = "sandbox_vector_index"
definicion = {"fields": [
    {"type": "vector", "path": "embedding", "numDimensions": dimensions(), "similarity": "cosine"},
    {"type": "filter", "path": "categoria"},
]}
if INDICE_SANDBOX not in search_indexes(sandbox):
    sandbox.create_search_index(SearchIndexModel(definition=definicion, name=INDICE_SANDBOX, type="vectorSearch"))
wait_until_queryable(sandbox, [INDICE_SANDBOX])
print("índice listo")

In [ ]:
def buscar_sandbox(consulta, k=3, filtro=None):
    etapa = {"index": INDICE_SANDBOX, "path": "embedding", "queryVector": embed_query(consulta),
             "numCandidates": 50, "limit": k}
    if filtro:
        etapa["filter"] = filtro
    return tabla(sandbox.aggregate([{"$vectorSearch": etapa},
                                    {"$project": {"_id": 0, "categoria": 1, "texto": 1,
                                                  "score": {"$meta": "vectorSearchScore"}}}]))


display(buscar_sandbox("¿cómo se genera electricidad con el sol?"))
display(buscar_sandbox("algo para el desayuno", filtro={"categoria": "cocina"}))

In [ ]:
# Consistencia eventual: inserta un texto nuevo y búscalo enseguida (puede tardar ~1 s en aparecer)
nuevo = "Las baterías de litio almacenan la energía solar para usarla de noche."
sandbox.insert_one({"categoria": "energía", "texto": nuevo, "embedding": embed_passages([nuevo])[0]})
for intento in range(20):
    if nuevo in buscar_sandbox("guardar energía para la noche", k=5)["texto"].tolist():
        print(f"visible en mongot después de ~{intento * 0.25:.2f} s")
        break
    time.sleep(0.25)

In [ ]:
# Limpieza (opcional): borra la colección sandbox y su índice
# db.drop_collection("sandbox")

## 9. RAG

Recupera fragmentos, arma el prompt y, si existe `.envs/anthropic.env` con `ANTHROPIC_API_KEY`, Claude responde
citándolos. Sin API key verás el prompt: es exactamente lo que recibiría el modelo.

### Juega aquí: pregúntale al libro

In [ ]:
pregunta = "¿Qué consejos le da don Quijote a Sancho antes de gobernar la ínsula?"
modo, k = "hybrid", 5          # prueba "semantic" o "text", y más o menos fragmentos

resultados = SEARCHES[modo](chunks, pregunta, k=k)
display(pd.DataFrame([{"#": i, "ref": reference(d), "capítulo": d["chapter_title"][:70]}
                      for i, d in enumerate(resultados, 1)]))
prompt = rag.build_prompt(pregunta, resultados)

if rag.has_credentials():
    rag.generate(prompt)                     # también acepta system=..., effort="low" | "high"
else:
    print("(Sin ANTHROPIC_API_KEY: este es el prompt que recibiría Claude)\n")
    print(prompt[:3000], "…" if len(prompt) > 3000 else "")

---

**¿Rompiste algo?** `python lessons/00_seed.py` reconstruye `quijote.chunks` y sus índices (y borra `sandbox`,
porque borra toda la base).